# 02. Score Quality

`papers_raw.csv`를 받아 정량 퀄리티 점수를 매깁니다. 

**점수 산식 (단순 가중합)**:
- 인용수 백분위 (해당 검색 결과 안에서) × 0.5
- 최신성 (2020 → 0.0, 2026 → 1.0 선형) × 0.3
- 저널/소스 보유 여부 × 0.1
- abstract 충분도 (200자 이상이면 1) × 0.1

이 점수는 **후보 좁히기를 위한 정량 신호**일 뿐입니다. 정성 평가는 `reference_quality_check(rr)` 프롬프트로 Claude에 맡기세요.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'papers_raw.csv')
print(f'{len(df)} papers loaded')
df.head()

In [ ]:
FROM_YEAR = 2020   # 01에서 사용한 값
TO_YEAR = 2026

def score(df, from_year=FROM_YEAR, to_year=TO_YEAR):
    cit = df['cited_by_count'].fillna(0).astype(float)
    if cit.max() == cit.min():
        cit_pct = pd.Series([0.5] * len(df), index=df.index)
    else:
        cit_pct = cit.rank(pct=True)

    year = df['year'].fillna(from_year).astype(float)
    recency = (year - from_year) / max(to_year - from_year, 1)
    recency = recency.clip(0, 1)

    has_venue = df['venue'].notna().astype(float)
    abs_ok = df['abstract'].fillna('').str.len().ge(200).astype(float)

    return (cit_pct * 0.5 + recency * 0.3 + has_venue * 0.1 + abs_ok * 0.1).round(3)

df['quality_score'] = score(df)
df_sorted = df.sort_values('quality_score', ascending=False).reset_index(drop=True)
df_sorted[['title', 'year', 'cited_by_count', 'venue', 'quality_score']].head(15)

In [ ]:
out = DATA_DIR / 'papers_scored.csv'
df_sorted.to_csv(out, index=False)
print(f'Saved → {out.resolve()}')

## 다음 단계

`03_export_to_claude.ipynb`로 상위 N개를 Claude Desktop에 붙여넣을 markdown 표로 export 하세요.